# ✈️ Flight Delays and Cancellations | Supervised Models

**Input:** `data/processed/flights_features.parquet`  
**Output:** trained models in `models/`

## Objective

Treinar e comparar três classificadores supervisionados para prever se um voo
será atrasado (`LABEL = 1`, atraso ≥ 15 min) ou pontual (`LABEL = 0`),
usando features temporais, de rota e históricas de atraso.

## Scope

- **Split temporal**: meses 1–10 para treino, meses 11–12 para teste —
  o modelo nunca vê dados futuros, simulando a condição real de predição.
- **Modelos avaliados**: Regressão Logística (baseline), Random Forest e
  Gradient Boosted Trees.
- **Desbalanceamento**: tratado via `weightCol` (LR e RF) e oversampling (GBT).
- **Métrica primária**: AUC-ROC — robusta ao desbalanceamento de classes.

## Setup

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from functools import reduce
from pyspark.sql import DataFrame
import pandas as pd

In [ ]:
os.environ['JAVA_HOME'] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = (
    SparkSession.builder
        .appName("Classifier")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.repl.eagerEval.enabled", True)
        .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("="*80)
print(f"✅ Spark Session iniciada ...\n\tSpark Version: {spark.version}\n\tPython Version: {sys.version}")
print("="*80)

## Data Loading

In [ ]:
df = spark.read.parquet('../data/processed/flights_features.parquet')
df.cache()

print(f'Registros : {df.count():,}')
print(f'Colunas   : {len(df.columns)}')
print('\nDistribuição do LABEL:')
df.groupBy('LABEL').count() \
.withColumn('PCT', F.round(F.col('count') / df.count() * 100, 2)) \
.orderBy('LABEL').show()

## Train / Test Split

In [ ]:
df_train = df.filter(F.col('MONTH') <= 10)
df_test  = df.filter(F.col('MONTH') >  10)

n_train = df_train.count()
n_test  = df_test.count()
total   = n_train + n_test

print(f'Treino : {n_train:>9,}  ({n_train/total*100:.1f}%)')
print(f'Teste  : {n_test:>9,}  ({n_test/total*100:.1f}%)')

print('\nDistribuição LABEL — Treino:')
df_train.groupBy('LABEL').count() \
        .withColumn('PCT', F.round(F.col('count') / n_train * 100, 2)) \
        .orderBy('LABEL').show()

print('Distribuição LABEL — Teste:')
df_test.groupBy('LABEL').count() \
       .withColumn('PCT', F.round(F.col('count') / n_test * 100, 2)) \
       .orderBy('LABEL').show()

## Class Imbalance

In [ ]:
# ── Class weights for imbalance ──────────────────────────────────────────────
class_counts = {
    row['LABEL']: row['count']
    for row in df_train.groupBy('LABEL').count().collect()
}
n_train_total = sum(class_counts.values())
class_weights = {
    label: n_train_total / (2 * count)
    for label, count in class_counts.items()
}

print('Pesos por classe:')
for k, v in sorted(class_weights.items()):
    print(f'  LABEL={k} → peso={v:.4f}')

df_train = df_train.withColumn(
    'CLASS_WEIGHT',
    F.when(F.col('LABEL') == 1, class_weights[1])
     .otherwise(class_weights[0])
     .cast(DoubleType())
)

# ── Feature set ───────────────────────────────────────────────────────────────
# ORIGIN_AIRPORT_IDX e DESTINATION_AIRPORT_IDX removidos:
# cardinalidade de 630 valores quebra o maxBins padrão das árvores,
# e o sinal já está capturado pelas features históricas HIST_DELAY_ORIGIN/DEST.

FEATURES = [
    # Temporais
    'MONTH', 'DAY_OF_WEEK', 'HORA_PARTIDA', 'HORA_CHEGADA_PROG',
    'IS_WEEKEND', 'IS_PEAK_MONTH',
    # Rota e distância
    'LOG_DISTANCE', 'FREQ_ROTA',
    # Categóricas indexadas (baixa cardinalidade)
    'AIRLINE_IDX', 'TURNO_IDX',
    # Históricas
    'HIST_DELAY_AIRLINE', 'HIST_DELAY_ORIGIN', 'HIST_DELAY_DEST',
    'HIST_DELAY_ROTA', 'HIST_DELAY_AIRLINE_DOW', 'HIST_DELAY_AIRLINE_TURNO',
]

print(f'\nTotal de features: {len(FEATURES)}')
print(FEATURES)

## Helper Functions

In [ ]:
def compute_metrics(predictions, model_name):
    binary_eval = BinaryClassificationEvaluator(
        labelCol='LABEL', rawPredictionCol='rawPrediction')
    multi_eval  = MulticlassClassificationEvaluator(
        labelCol='LABEL', predictionCol='prediction')

    metrics = {
        'Model'    : model_name,
        'AUC-ROC'  : round(binary_eval.evaluate(predictions,
                        {binary_eval.metricName: 'areaUnderROC'}), 4),
        'F1'       : round(multi_eval.evaluate(predictions,
                        {multi_eval.metricName: 'f1'}), 4),
        'Precision': round(multi_eval.evaluate(predictions,
                        {multi_eval.metricName: 'weightedPrecision'}), 4),
        'Recall'   : round(multi_eval.evaluate(predictions,
                        {multi_eval.metricName: 'weightedRecall'}), 4),
        'Accuracy' : round(multi_eval.evaluate(predictions,
                        {multi_eval.metricName: 'accuracy'}), 4),
    }

    print(f'\n── {model_name} ──────────────────────')
    for k, v in metrics.items():
        if k != 'Model':
            print(f'  {k:<12}: {v}')
    return metrics


def confusion_matrix_display(predictions, model_name):
    cm = (
        predictions
        .groupBy('LABEL', 'prediction')
        .count()
        .toPandas()
        .pivot(index='LABEL', columns='prediction', values='count')
        .fillna(0).astype(int)
    )
    cm.index   = [f'Real {int(i)}' for i in cm.index]
    cm.columns = [f'Pred {int(c)}' for c in cm.columns]
    print(f'\nMatriz de Confusão — {model_name}:')
    print(cm.to_string())

## Supervised Models

O desbalanceamento das classes (≈ 81% pontual / 19% atrasado) é tratado de
forma diferente por modelo:
- **Regressão Logística e Random Forest**: parâmetro `weightCol` — penaliza
  erros na classe minoritária proporcionalmente.
- **GBT**: não suporta `weightCol` — oversampling da classe minoritária
  até atingir proporção 1:1.

### 1. Logistic Regression (Baseline)

In [ ]:
assembler = VectorAssembler(
    inputCols=FEATURES, outputCol='features_raw', handleInvalid='keep')

scaler = StandardScaler(
    inputCol='features_raw', outputCol='features',
    withMean=True, withStd=True)

lr = LogisticRegression(
    featuresCol='features', labelCol='LABEL', weightCol='CLASS_WEIGHT',
    maxIter=100, regParam=0.01, elasticNetParam=0.0)

pipeline_lr = Pipeline(stages=[assembler, scaler, lr])

print('Treinando Regressão Logística...')
model_lr = pipeline_lr.fit(df_train)
print('Concluído ✅')

results = []
preds_lr = model_lr.transform(df_test)
metrics_lr = compute_metrics(preds_lr, 'Logistic Regression (baseline)')
results.append(metrics_lr)
confusion_matrix_display(preds_lr, 'Logistic Regression (baseline)')

### 2. Random Forest

In [ ]:
rf = RandomForestClassifier(
    featuresCol='features_raw', labelCol='LABEL', weightCol='CLASS_WEIGHT',
    numTrees=100, maxDepth=10, maxBins=50,
    minInstancesPerNode=20, featureSubsetStrategy='sqrt', seed=42)

pipeline_rf = Pipeline(stages=[assembler, rf])

print('Treinando Random Forest...')
model_rf = pipeline_rf.fit(df_train)
print('Concluído ✅')

preds_rf = model_rf.transform(df_test)
metrics_rf = compute_metrics(preds_rf, 'Random Forest')
results.append(metrics_rf)
confusion_matrix_display(preds_rf, 'Random Forest')

### 3. Gradient Boosted Trees (GBT)

In [ ]:
# ── GBT does not support weightCol — oversample minority class ───────────────
oversample_ratio = round(class_counts[0] / class_counts[1])
print(f'Ratio de oversampling: {oversample_ratio}x')

df_minority       = df_train.filter(F.col('LABEL') == 1)
df_majority       = df_train.filter(F.col('LABEL') == 0)
df_train_balanced = reduce(
    DataFrame.union,
    [df_majority] + [df_minority] * oversample_ratio
).orderBy(F.rand(seed=42))

print(f'Dataset balanceado: {df_train_balanced.count():,} registros')

gbt = GBTClassifier(
    featuresCol='features_raw', labelCol='LABEL',
    maxIter=100, maxDepth=6, stepSize=0.1,
    subsamplingRate=0.8, maxBins=50, seed=42)

pipeline_gbt = Pipeline(stages=[assembler, gbt])

print('Treinando GBT...')
model_gbt = pipeline_gbt.fit(df_train_balanced)
print('Concluído ✅')

preds_gbt = model_gbt.transform(df_test)
metrics_gbt = compute_metrics(preds_gbt, 'Gradient Boosted Trees')
results.append(metrics_gbt)
confusion_matrix_display(preds_gbt, 'Gradient Boosted Trees')

## Model Comparison

Resumo das métricas de todos os modelos no conjunto de teste (meses 11-12).
O **AUC-ROC** é a métrica primária por ser robusta ao desbalanceamento;
**F1** equilibra precisão e recall na classe positiva (atraso).

In [ ]:
df_results = pd.DataFrame(results).set_index('Model')
print('\n' + '='*60)
print('Comparação final dos modelos:')
print('='*60)
print(df_results.to_string())

best_model = df_results['AUC-ROC'].idxmax()
print(f'\n🏆 Melhor modelo (AUC-ROC): {best_model}')

## Feature Importance

Random Forest e GBT expõem o vetor de importância de cada feature.
Valores maiores indicam maior contribuição para a separação das classes.
Essa análise ajuda a identificar features redundantes e orientar
melhorias no pipeline de feature engineering.

In [ ]:
def plot_feature_importance(model_pipeline, model_name):
    importances = model_pipeline.stages[-1].featureImportances.toArray()
    df_importance = (
        pd.DataFrame({'feature': FEATURES, 'importance': importances})
        .sort_values('importance', ascending=False)
    )
    print(f'\nImportância das features — {model_name}:')
    print(df_importance.to_string(index=False))
    return df_importance

importance_rf  = plot_feature_importance(model_rf,  'Random Forest')
importance_gbt = plot_feature_importance(model_gbt, 'GBT')

## Model Persistence

Os três modelos são salvos no formato nativo do Spark ML (`PipelineModel`)
para uso posterior em inferência ou deploy.

In [ ]:
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

models_to_save = [
    (model_lr,  'logistic_regression'),
    (model_rf,  'random_forest'),
    (model_gbt, 'gbt'),
]

for model, name in models_to_save:
    path = f'{MODELS_DIR}/{name}'
    model.write().overwrite().save(path)
    print(f'✅ {name:<22} salvo em {path}')

print('\nTodos os modelos persistidos com sucesso.')